# PaySim - Data Profiling

Analysis of transaction structure, data quality, fraud prevalence, transaction amounts, activity over simulation steps, and balance consistency in the local PaySim dataset.

All summary tables and quality checks cover the complete dataset. The origin-balance scatter plot uses a random sample of up to 30,000 transactions with seed 42. Amounts retain their original, unverified currency unit. Step values represent simulation activity rather than calendar timestamps.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True})
import json

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'configs/data_profiling.json').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Project configuration not found: configs/data_profiling.json')
CONFIG_PATH = PROJECT_ROOT / 'configs/data_profiling.json'
CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
SCHEMA_PATH = PROJECT_ROOT / CONFIG['dataset']['schema_path']
SCHEMA = json.loads(SCHEMA_PATH.read_text(encoding='utf-8'))
RANDOM_STATE = CONFIG['visualization']['random_seed']
plt.rcParams['figure.dpi'] = CONFIG['visualization']['figure_dpi']


FileNotFoundError: Project configuration not found: configs/data_profiling.json

## 1. Dataset Identity and Structure

The input snapshot is identified by its file size and SHA-256 checksum. The CSV contains 11 columns describing transaction types, amounts, account identifiers, balances, fraud labels, and the existing system flag.

`isFraud` is the target label; `isFlaggedFraud` is a separate system flag. Account identifiers are categorical identifiers. The original EDA describes `step` as a simulated hour and merchant destination balances as potentially unrecorded. These interpretations remain subject to source verification. Zero balances alone do not establish missing data.


In [ ]:
import os
import hashlib
import platform
from datetime import datetime, timezone

data_override = os.environ.get(CONFIG['dataset']['path_environment_variable'])
DATA_PATH = (Path(data_override) if data_override else PROJECT_ROOT / CONFIG['dataset']['path']).resolve()
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')
ROOT = PROJECT_ROOT
REPORT_DIR = ROOT / CONFIG['output']['report_directory']
OUTPUT_DIR = ROOT / CONFIG['output']['tables_directory']
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sha = hashlib.sha256()
with DATA_PATH.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        sha.update(block)
fingerprint = sha.hexdigest()
# Infer numeric types first: narrowing integers before validation can hide invalid values.
df = pd.read_csv(DATA_PATH, dtype=CONFIG['csv']['dtype_overrides'],
                 encoding=CONFIG['csv']['encoding'], sep=CONFIG['csv']['separator'],
                 on_bad_lines=CONFIG['csv']['malformed_rows'])
expected_columns = [column['name'] for column in SCHEMA['columns']]
if df.columns.tolist() != expected_columns:
    raise ValueError(f'Unexpected schema: {df.columns.tolist()}')
numeric_columns = [c['name'] for c in SCHEMA['columns'] if c['kind'] in ['integer', 'number']]
if not all(pd.api.types.is_numeric_dtype(df[c]) for c in numeric_columns):
    raise ValueError('Non-numeric values in numeric fields: inspect raw CSV before profiling.')
manifest = {
    'file_name': DATA_PATH.name, 'sha256': fingerprint, 'file_size_bytes': DATA_PATH.stat().st_size,
    'row_count': len(df), 'columns': expected_columns, 'scope': 'full dataset',
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(), 'pandas': pd.__version__,
    'numpy': np.__version__, 'matplotlib': plt.matplotlib.__version__,
    'download_url': None, 'publisher_version': None, 'currency_unit': CONFIG['dataset']['currency_unit'],
    'step_unit': CONFIG['dataset']['step_unit'],
    'config_sha256': hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),
    'schema_sha256': hashlib.sha256(SCHEMA_PATH.read_bytes()).hexdigest(),
    'reference_snapshot_matches': fingerprint == CONFIG['dataset']['reference_sha256'],
    'provenance_status': 'Pending confirmation; see docs/data-source.md',
}
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns; SHA-256: {fingerprint}')
print(f'DataFrame memory: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB')
display(df.head())

## 2. Structure and Data Quality

Column types, numerical summaries, missing values, exact duplicate rows, and label domains characterize the structure and basic integrity of the complete dataset. Duplicate counts refer to repeated rows beyond their first occurrence.


In [ ]:
df.info()
display(df.describe(include='number').T)
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100,
})
display(quality)
print(f'Exact duplicate rows: {df.duplicated().sum():,}')
print(f'Negative transaction amounts: {(df["amount"] < 0).sum():,}')
for column in ['isFraud', 'isFlaggedFraud']:
    print(f'{column} values: {sorted(df[column].dropna().unique().tolist())}')


## 3. Fraud Class Balance

Label counts and percentages quantify class imbalance. The fraud rate describes transaction prevalence, while overall accuracy can conceal poor detection of the minority class.


In [ ]:
class_summary = df.groupby('isFraud').size().rename('count').to_frame()
class_summary['percent'] = class_summary['count'] / len(df) * 100
display(class_summary)
fraud_rate = df['isFraud'].mean() * 100
print(f'Overall fraud rate: {fraud_rate:.4f}%')

ax = class_summary['count'].rename(index={0: 'Non-fraud', 1: 'Fraud'}).plot.bar(color=['steelblue', 'tomato'])
ax.set(title='Transaction Counts by Fraud Label', xlabel='Label', ylabel='Number of transactions')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()


## 4. Transaction Types

Transaction volume, fraud count, fraud rate, median amount, and total amount summarize differences between transaction categories. Volume and fraud rate measure different aspects of each category.


In [ ]:
type_summary = df.groupby('type', observed=True).agg(
    transactions=('isFraud', 'size'),
    fraud_count=('isFraud', 'sum'),
    fraud_rate=('isFraud', 'mean'),
    median_amount=('amount', 'median'),
    total_amount=('amount', 'sum'),
).sort_values('transactions', ascending=False)
type_summary['fraud_rate_percent'] = type_summary.pop('fraud_rate') * 100
display(type_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
type_summary['transactions'].plot.bar(ax=axes[0], color='steelblue')
axes[0].set(title='Transaction Volume by Type', xlabel='Type', ylabel='Transactions')
type_summary['fraud_rate_percent'].plot.bar(ax=axes[1], color='tomato')
axes[1].set(title='Fraud Rate by Type', xlabel='Type', ylabel='Fraud rate (%)')
plt.tight_layout()
plt.show()


## 5. Transaction Amounts

Quantiles describe the long-tailed amount distribution. Histograms compare fraud and non-fraud transactions on a log(1 + amount) scale. Negative amounts are counted in the quality checks and excluded from these histograms.


In [ ]:
display(df.groupby('isFraud')['amount'].describe(percentiles=CONFIG['profiling']['amount_percentiles_by_label']).T)

fig, ax = plt.subplots()
for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    amounts = df.loc[(df['isFraud'] == label) & (df['amount'] >= 0), 'amount']
    ax.hist(np.log1p(amounts), bins=CONFIG['visualization']['amount_histogram_bins'], density=True, alpha=0.5,
            label='Fraud' if label else 'Non-fraud', color=color)
ax.set(title='Transaction Amount Distribution', xlabel='log(1 + amount)', ylabel='Density')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Activity over Simulation Steps

Transaction counts and fraud rates are aggregated by step. Low-volume steps can have high fraud rates despite small absolute fraud counts. The horizontal axis uses simulation steps rather than real calendar timestamps.


In [ ]:
hourly = df.groupby('step').agg(
    transactions=('isFraud', 'size'), fraud_count=('isFraud', 'sum'),
    fraud_rate=('isFraud', 'mean'),
)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(hourly.index, hourly['transactions'], color='steelblue')
axes[0].set(title='Transaction Activity over Time', ylabel='Transactions')
axes[1].plot(hourly.index, hourly['fraud_rate'] * 100, color='tomato')
axes[1].set(xlabel='Simulation step (hour)', ylabel='Fraud rate (%)')
plt.tight_layout()
plt.show()


## 7. Balances and Numeric Relationships

Pearson correlation summarizes linear association among numeric fields. Account identifiers are excluded. Correlation does not establish causation, and the rare fraud label affects interpretation.

New balances and balance-derived differences may contain information recorded after the transaction. Their availability depends on the scoring time. The scatter plot uses a fixed random sample; the correlation matrix covers the complete dataset.


In [ ]:
numeric_columns = [c['name'] for c in SCHEMA['columns'] if c['kind'] in ['integer', 'number']]
corr = df[numeric_columns].corr()
display(corr['isFraud'].sort_values(ascending=False).to_frame('correlation_with_fraud'))

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title('Numeric Correlation Matrix')
fig.colorbar(im, ax=ax, label='Pearson correlation')
plt.tight_layout()
plt.show()

# Sample only for visualization; all summary tables use the complete dataset.
sample = df.sample(n=min(CONFIG['visualization']['scatter_sample_rows'], len(df)), random_state=RANDOM_STATE)
fig, ax = plt.subplots()
for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    subset = sample.loc[sample['isFraud'] == label]
    ax.scatter(subset['oldbalanceOrg'], subset['amount'], s=12, alpha=0.35,
               color=color, label='Fraud' if label else 'Non-fraud')
ax.set_xscale('symlog', linthresh=1)
ax.set_yscale('symlog', linthresh=1)
ax.set(title='Origin Balance vs. Amount (Random Sample)',
       xlabel='Origin balance before transaction (symlog)', ylabel='Amount (symlog)')
ax.legend()
plt.tight_layout()
plt.show()


## 8. Existing Fraud Flag

The confusion matrix compares the existing system flag with the fraud label. Precision is the fraction of flagged transactions labeled as fraud; recall is the fraction of labeled fraud transactions flagged by the system. These statistics describe the existing rule rather than a trained model.


In [ ]:
flag_table = pd.crosstab(df['isFraud'], df['isFlaggedFraud']).reindex(
    index=[0, 1], columns=[0, 1], fill_value=0)
display(flag_table.rename_axis(index='Actual fraud', columns='System flag'))
tn, fp = flag_table.loc[0, 0], flag_table.loc[0, 1]
fn, tp = flag_table.loc[1, 0], flag_table.loc[1, 1]
precision = tp / (tp + fp) if tp + fp else np.nan
recall = tp / (tp + fn) if tp + fn else np.nan
display(pd.Series({'precision': precision, 'recall': recall}, name='Existing flag metrics'))


## 9. Column Profiles and Domain Checks

Per-column profiles include data type, missingness, cardinality, blank strings, zeros, negative values, and nonfinite numeric values. Domain checks cover required fields, positive integer steps, nonnegative amounts, binary labels, and the five expected transaction categories.

A row may violate multiple rules. Rule counts overlap, whereas the count of rows failing at least one rule counts each affected row once. Exact duplicate counts exclude the first occurrence. IQR outliers describe unusual amount values; they are not fraud labels. The analysis reports these conditions without modifying the raw data.


In [ ]:
column_profile = quality.copy()
column_profile['unique_count'] = df.nunique(dropna=True)
column_profile['blank_count'] = 0
for c in SCHEMA['domain_checks']['blank_string_columns']:
    column_profile.loc[c, 'blank_count'] = int(df[c].astype('string').str.strip().eq('').sum())
column_profile['zero_count'] = pd.Series({c: int(df[c].eq(0).sum()) for c in numeric_columns})
column_profile['negative_count'] = pd.Series({c: int(df[c].lt(0).sum()) for c in numeric_columns})
column_profile['nonfinite_count'] = pd.Series({c: int((df[c].notna() & ~np.isfinite(df[c])).sum()) for c in numeric_columns})
display(column_profile)
numeric_summary = df[numeric_columns].describe(percentiles=CONFIG['profiling']['numeric_percentiles']).T
display(numeric_summary)
duplicate_rows = int(df.duplicated().sum())
invalid_any = np.zeros(len(df), dtype=bool)
rules = []
def add_rule(name, mask, policy):
    global invalid_any
    values = pd.Series(mask).fillna(False).to_numpy(dtype=bool)
    invalid_any |= values
    rules.append({'rule': name, 'count': int(values.sum()),
                  'percent': float(values.mean() * 100), 'proposed_policy': policy})

add_rule('required_null', df.isna().any(axis=1), 'Quarantine; preserve source row and reasons')
for c in SCHEMA['domain_checks']['blank_string_columns']:
    add_rule(f'{c}_blank', df[c].astype('string').str.strip().eq(''), 'Quarantine; do not invent identifiers/categories')
for c in numeric_columns:
    add_rule(f'{c}_nonfinite', df[c].notna() & ~np.isfinite(df[c]), 'Quarantine; do not silently cast')
add_rule('step_nonpositive_or_fractional', df.step.lt(SCHEMA['domain_checks']['step_minimum']) | df.step.mod(1).ne(0), 'Quarantine; step must be a positive integer')
add_rule('amount_negative', df.amount.lt(SCHEMA['domain_checks']['amount_minimum']), 'Quarantine pending domain review')
for c in SCHEMA['domain_checks']['label_values']:
    add_rule(f'{c}_outside_binary', df[c].notna() & ~df[c].isin(SCHEMA['domain_checks']['label_values'][c]), 'Quarantine; expected 0/1')
add_rule('unknown_type', df.type.notna() & ~df.type.isin(SCHEMA['domain_checks']['transaction_types']),
         'Quarantine/review contract; do not drop silently')
rule_summary = pd.DataFrame(rules)
print(f'Rows failing at least one proposed rule: {invalid_any.sum():,}')
print(f'Exact duplicate rows beyond first occurrence: {duplicate_rows:,}')
display(rule_summary)
q1, q3 = df.amount.quantile([.25, .75])
iqr = q3 - q1
amount_outliers = int((df.amount.lt(q1 - CONFIG['profiling']['iqr_multiplier'] * iqr) | df.amount.gt(q3 + CONFIG['profiling']['iqr_multiplier'] * iqr)).sum())
print(f'Zero amounts: {df.amount.eq(0).sum():,}; IQR outliers: {amount_outliers:,} (retain for review)')

## 10. Balance Consistency and Account Identifiers

Balance residuals use an absolute tolerance of 0.01 in the original amount unit. The origin residual is `new balance - old balance - sign * amount`, with sign +1 for CASH_IN and -1 for the other transaction types.

Destination residuals cover CASH_IN (-1), CASH_OUT (+1), and TRANSFER (+1) for C-prefixed accounts with at least one nonzero destination balance. Merchant accounts, other transaction types, and records with both destination balances equal to zero are excluded from destination checks and counted separately.

These formulas are diagnostic assumptions. Residual mismatches can reflect balance coverage or simulation behavior and do not establish fraud. Account cardinality and identifier prefixes characterize the identifier fields; the step summary describes temporal coverage and source ordering.


In [ ]:
balance_rows = []
for transaction_type in type_summary.index:
    part = df.loc[df.type.eq(transaction_type)]
    origin_sign = CONFIG['balance_checks']['origin_amount_sign'][str(transaction_type)]
    origin_error = part.newbalanceOrig - part.oldbalanceOrg - origin_sign * part.amount
    dest_sign = CONFIG['balance_checks']['destination_amount_sign'].get(str(transaction_type))
    eligible = part.nameDest.str.startswith(CONFIG['balance_checks']['destination_account_prefix']).fillna(False)
    if CONFIG['balance_checks']['exclude_both_destination_balances_zero']:
        eligible &= part.oldbalanceDest.ne(0) | part.newbalanceDest.ne(0)
    if dest_sign is None:
        eligible = pd.Series(False, index=part.index)
    dest_error = part.newbalanceDest - part.oldbalanceDest - (dest_sign or 0) * part.amount
    balance_rows.append({'type': str(transaction_type), 'rows': len(part),
        'origin_mismatch': int(origin_error.abs().gt(CONFIG['balance_checks']['absolute_tolerance']).sum()),
        'destination_eligible': int(eligible.sum()),
        'destination_mismatch': int((eligible & dest_error.abs().gt(CONFIG['balance_checks']['absolute_tolerance'])).sum()),
        'destination_excluded': int((~eligible).sum()),
        'both_destination_balances_zero': int((part.oldbalanceDest.eq(0) & part.newbalanceDest.eq(0)).sum())})
balance_summary = pd.DataFrame(balance_rows).set_index('type')
display(balance_summary)
account_summary = pd.DataFrame({
    'distinct_accounts': [df.nameOrig.nunique(), df.nameDest.nunique()],
    'customer_prefix_rows': [df.nameOrig.str.startswith('C').sum(), df.nameDest.str.startswith('C').sum()],
    'merchant_prefix_rows': [df.nameOrig.str.startswith('M').sum(), df.nameDest.str.startswith('M').sum()],
}, index=['origin', 'destination'])
display(account_summary)
step_summary = hourly.copy()
step_summary['fraud_rate_percent'] = step_summary.pop('fraud_rate') * 100
print(f'Step range: {df.step.min()}-{df.step.max()}; observed steps: {df.step.nunique():,}')
print(f'Source ordered by step: {df.step.is_monotonic_increasing}')

## 11. Data Quality by Transaction Type and Fraud Label

Counts and within-group percentages describe how zero amounts, zero balances, amount outliers,
and domain violations vary across transaction types and fraud labels. Amount outliers use the
full-dataset IQR bounds; these are descriptive, not fitted model preprocessing parameters.

Origin mismatch rates use records with finite origin balances and amounts and a defined transaction sign.
Destination mismatch rates use eligible destination records as their denominator, following section 10.
Other condition percentages use all transactions in the corresponding type-label group.
Conditions overlap and do not independently establish fraud.

In [ ]:
quality_events = df[['type', 'isFraud']].copy()
quality_events['transactions'] = 1
quality_events['zero_amount'] = df.amount.eq(0)
quality_events['zero_origin_balances'] = df.oldbalanceOrg.eq(0) & df.newbalanceOrig.eq(0)
quality_events['zero_destination_balances'] = df.oldbalanceDest.eq(0) & df.newbalanceDest.eq(0)
amount_outlier_mask = (df.amount.lt(q1 - CONFIG['profiling']['iqr_multiplier'] * iqr)
                       | df.amount.gt(q3 + CONFIG['profiling']['iqr_multiplier'] * iqr))
quality_events['amount_iqr_outlier'] = amount_outlier_mask
quality_events['domain_violation'] = invalid_any
origin_signs = df.type.astype('string').map(CONFIG['balance_checks']['origin_amount_sign'])
dest_signs = df.type.astype('string').map(CONFIG['balance_checks']['destination_amount_sign'])
origin_eligible = np.isfinite(df[['amount', 'oldbalanceOrg', 'newbalanceOrig']]).all(axis=1) & origin_signs.notna()
dest_eligible = (np.isfinite(df[['amount', 'oldbalanceDest', 'newbalanceDest']]).all(axis=1)
                 & dest_signs.notna()
                 & df.nameDest.str.startswith(CONFIG['balance_checks']['destination_account_prefix']).fillna(False))
if CONFIG['balance_checks']['exclude_both_destination_balances_zero']:
    dest_eligible &= df.oldbalanceDest.ne(0) | df.newbalanceDest.ne(0)
tolerance = CONFIG['balance_checks']['absolute_tolerance']
quality_events['origin_eligible'] = origin_eligible
quality_events['origin_mismatch'] = origin_eligible & (df.newbalanceOrig - df.oldbalanceOrg - origin_signs * df.amount).abs().gt(tolerance)
quality_events['destination_eligible'] = dest_eligible
quality_events['destination_mismatch'] = dest_eligible & (df.newbalanceDest - df.oldbalanceDest - dest_signs * df.amount).abs().gt(tolerance)
quality_by_group = quality_events.groupby(['type', 'isFraud'], observed=True, dropna=False).sum()
for condition in ['zero_amount', 'zero_origin_balances', 'zero_destination_balances', 'amount_iqr_outlier', 'domain_violation']:
    quality_by_group[f'{condition}_percent'] = quality_by_group[condition] / quality_by_group.transactions * 100
for prefix in ['origin', 'destination']:
    quality_by_group[f'{prefix}_mismatch_percent'] = (
        quality_by_group[f'{prefix}_mismatch'] / quality_by_group[f'{prefix}_eligible'].replace(0, np.nan) * 100)
display(quality_by_group)
assert int(quality_by_group.transactions.sum()) == len(df)
assert int(quality_by_group.amount_iqr_outlier.sum()) == amount_outliers
del quality_events, origin_signs, dest_signs, origin_eligible, dest_eligible

## 12. Temporal Split Feasibility

The exploratory train, validation, and test windows target 70%, 15%, and 15% of transaction rows,
respectively, using the configured fractions. Cutoffs occur at whole simulation steps and are chosen
from cumulative transaction counts. Entire steps stay in a single window, so realized fractions can
differ from the targets. Label counts and prevalence characterize each window; no model is fitted.

Account overlap counts distinct identifiers appearing in both windows, separately for origin and
destination roles. The percentage denominator is the distinct account count in the later window.
Repeated accounts can occur under chronological evaluation and do not alone establish leakage.
The analysis quantifies temporal coverage and entity reuse; these cutoffs are not a finalized training split.

In [ ]:
split_names = CONFIG['temporal_split']['split_names']
fractions = np.asarray(CONFIG['temporal_split']['target_row_fractions'], dtype=float)
if len(split_names) != 3 or len(fractions) != 3 or (fractions <= 0).any() or not np.isclose(fractions.sum(), 1):
    raise ValueError('Temporal feasibility requires three positive fractions totaling one.')
step_counts = df.groupby('step', dropna=False).size().sort_index()
if df.step.isna().any() or len(step_counts) < 3:
    raise ValueError('Temporal feasibility requires non-null steps and at least three distinct steps.')
cumulative = step_counts.cumsum()
cutoff_positions = [int(cumulative.searchsorted(len(df) * share, side='left'))
                    for share in np.cumsum(fractions)[:2]]
train_end, validation_end = [step_counts.index[position] for position in cutoff_positions]
if not train_end < validation_end < step_counts.index[-1]:
    raise ValueError('Row targets do not yield three non-empty whole-step windows.')
split_id = pd.Series(np.select([df.step.le(train_end), df.step.le(validation_end)], [0, 1], default=2), index=df.index)
split_rows = []
for split_number, split_name in enumerate(split_names):
    mask = split_id.eq(split_number)
    labels = df.loc[mask, 'isFraud']
    split_rows.append({'split': split_name, 'step_min': df.loc[mask, 'step'].min(),
                      'step_max': df.loc[mask, 'step'].max(), 'transactions': int(mask.sum()),
                      'row_percent': float(mask.mean() * 100), 'target_percent': fractions[split_number] * 100,
                      'fraud_count': int(labels.eq(1).sum()), 'nonfraud_count': int(labels.eq(0).sum()),
                      'fraud_percent': float(labels.eq(1).mean() * 100),
                      'both_classes_present': bool(labels.eq(1).any() and labels.eq(0).any())})
split_summary = pd.DataFrame(split_rows).set_index('split')
display(split_summary)
assert int(split_summary.transactions.sum()) == len(df)
assert split_id.groupby(df.step).nunique().eq(1).all()
assert split_summary.step_max.iloc[0] < split_summary.step_min.iloc[1] < split_summary.step_min.iloc[2]
overlap_rows = []
for account_column in ['nameOrig', 'nameDest']:
    account_sets = {name: pd.Index(df.loc[split_id.eq(number), account_column].dropna().unique())
                    for number, name in enumerate(split_names)}
    for left_number, right_number in [(0, 1), (0, 2), (1, 2)]:
        left, right = split_names[left_number], split_names[right_number]
        overlap = len(account_sets[left].intersection(account_sets[right], sort=False))
        right_count = len(account_sets[right])
        overlap_rows.append({'account_role': account_column, 'earlier_split': left, 'later_split': right,
                             'earlier_distinct_accounts': len(account_sets[left]),
                             'later_distinct_accounts': right_count, 'shared_distinct_accounts': overlap,
                             'later_accounts_seen_in_earlier_percent': overlap / right_count * 100 if right_count else np.nan})
    del account_sets
account_overlap = pd.DataFrame(overlap_rows)
display(account_overlap)
temporal_manifest = {'status': CONFIG['temporal_split']['status'], 'target_row_fractions': fractions.tolist(),
                     'train_end_step': int(train_end), 'validation_end_step': int(validation_end),
                     'whole_step_boundaries': True, 'account_overlap_scope': 'same role, pairwise windows'}

## 13. Key Findings

The following findings are calculated from the complete input snapshot. They summarize label imbalance,
fraud concentration, quality conditions, the existing system flag, and the exploratory time windows.
An outlier percentage among fraud transactions describes association, not detection performance.

In [ ]:
fraud_count = int(df.isFraud.eq(1).sum())
fraud_types = type_summary.loc[type_summary.fraud_count.gt(0)].index.astype(str).tolist()
findings = [
    ('Dataset size', f'{len(df):,} transactions; {df.shape[1]} columns'),
    ('Fraud prevalence', f'{fraud_count:,} transactions ({fraud_rate:.4f}%)'),
    ('Types containing fraud', ', '.join(fraud_types) or 'None'),
    ('Largest transaction category', str(type_summary.transactions.idxmax())),
    ('Highest observed type-level fraud rate', f'{type_summary.fraud_rate_percent.idxmax()}: {type_summary.fraud_rate_percent.max():.4f}%'),
    ('Missing cells / duplicate rows', f'{int(df.isna().sum().sum()):,} / {duplicate_rows:,}'),
    ('Rows failing proposed domain checks', f'{int(invalid_any.sum()):,}'),
    ('Zero-amount transactions', f'{int(df.amount.eq(0).sum()):,}'),
    ('Amount IQR outliers', f'{amount_outliers:,} ({amount_outliers / len(df) * 100:.4f}%)'),
    ('IQR outlier fraction among fraud', f'{(amount_outlier_mask & df.isFraud.eq(1)).sum() / fraud_count * 100:.4f}%' if fraud_count else 'Undefined: no fraud rows'),
    ('Existing flag precision / recall', f'{precision:.6f} / {recall:.6f}'),
    ('Fraud not captured by existing flag', f'{int(fn):,} of {fraud_count:,}'),
    ('Exploratory step boundaries', f'train <= {train_end}; validation <= {validation_end}; test > {validation_end}'),
    ('Both classes in every exploratory window', str(bool(split_summary.both_classes_present.all()))),
]
key_findings = pd.DataFrame(findings, columns=['finding', 'observed_result']).set_index('finding')
display(key_findings)

## 14. Interpretation and Limitations

The label distribution measures fraud prevalence in this local snapshot. Type-level and step-level fraud rates are conditional distributions, not estimates of model performance. The existing flag provides a separate rule baseline.

Zero amounts, zero balances, duplicates, IQR outliers, and balance residuals represent different data conditions. Their counts do not imply a common rejection rule. In particular, unrecorded destination balances can affect residual checks without producing null values.

`isFraud` contains the target outcome. `isFlaggedFraud` contains a system decision. New balances and balance residuals may reveal information available only after a transaction. Account identifiers have high cardinality and represent entities rather than continuous measurements. These distinctions affect how associations with fraud are interpreted.

Download provenance, publisher version, license, currency unit, and step semantics remain unverified. The dataset is described as synthetic in the original EDA; its patterns and fraud rates do not establish behavior or detection performance in real banking transactions.


In [ ]:
import json

tables = {'column_profile': column_profile, 'numeric_summary': numeric_summary,
          'quality_rules': rule_summary, 'class_distribution': class_summary,
          'transaction_types': type_summary, 'step_distribution': step_summary,
          'balance_checks': balance_summary, 'account_summary': account_summary,
          'flag_confusion': flag_table, 'quality_by_type_and_label': quality_by_group,
          'temporal_split_feasibility': split_summary, 'account_overlap': account_overlap,
          'key_findings': key_findings}
for name, table in tables.items():
    table.to_csv(OUTPUT_DIR / f'{name}.csv', index=True)
manifest.update({'temporal_split_feasibility': temporal_manifest, 'duplicate_rows': duplicate_rows, 'invalid_rows_proposed_rules': int(invalid_any.sum()),
                 'fraud_count': int(df.isFraud.eq(1).sum()), 'step_min': int(df.step.min()),
                 'step_max': int(df.step.max()), 'amount_iqr_outliers': amount_outliers})
(OUTPUT_DIR / CONFIG['output']['manifest_filename']).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
lines = [
    '# PaySim data quality report', '',
    'Generated by `notebooks/01_data_profiling.ipynb`; all checks use the full local CSV.', '',
    f'- Run UTC: {manifest["run_utc"]}', f'- SHA-256: `{fingerprint}`',
    f'- File size: {manifest["file_size_bytes"]:,} bytes',
    f'- Rows: {len(df):,}; columns: {df.shape[1]}',
    f'- Fraud: {manifest["fraud_count"]:,} ({fraud_rate:.4f}%)',
    f'- Step range: {manifest["step_min"]}-{manifest["step_max"]}',
    f'- Missing cells: {int(df.isna().sum().sum()):,}',
    f'- Duplicate rows beyond first occurrence: {duplicate_rows:,}; retain pending review',
    f'- Rows failing one or more proposed rules: {int(invalid_any.sum()):,}',
    f'- Zero amounts: {int(df.amount.eq(0).sum()):,}; retain pending review',
    f'- Amount IQR outliers: {amount_outliers:,}; retain, not automatically fraud',
    f'- Exploratory time cutoffs: train <= {train_end}; validation <= {validation_end}; test > {validation_end}',
    f'- Existing flag precision: {precision:.6f}; recall: {recall:.6f} (rule baseline only)',
    '', '## Rules and proposed policy', '',
    '| Rule | Rows | Proposed policy |', '|---|---:|---|',
]
lines += [f'| {r.rule} | {r.count:,} | {r.proposed_policy} |' for r in rule_summary.itertuples(index=False)]
lines += ['', 'Rules overlap; counts must not be summed to calculate quarantine rows.',
          '', '## Balance checks (absolute tolerance 0.01)', '',
          '| Type | Origin mismatches | Destination eligible | Destination mismatches | Destination excluded |',
          '|---|---:|---:|---:|---:|']
lines += [f'| {t} | {r.origin_mismatch:,} | {r.destination_eligible:,} | {r.destination_mismatch:,} | {r.destination_excluded:,} |'
          for t, r in balance_summary.iterrows()]
lines += ['', 'Balance formulas and destination coverage are described in notebook section 10. '
          'These counts are investigation signals, not automatic fraud labels or rejection rules.',
          '', '## Handoff and open decisions', '',
          '- MVP candidates: amount/log1p(amount) and type; review step/old balances before inclusion.',
          '- Exclude label, existing flag, new balances and balance residuals from default scoring features.',
          '- Preserve account IDs for traceability; propose snapshot/file/source-row identity at ingestion.',
          '- Source URL/version/license, currency, step semantics: Project Lead / Batch and ML Lead, by end of W1.',
          '- Label/feature availability: Evaluation Lead; scoring fields: Streaming Lead; validation policy: Quality Assurance Lead, by end of W1.',
          '- No calendar deadline, scoring contract, timestamp anchor or ETL policy has been approved here.',
          '- Raw data is retained unchanged; no imputation, clipping, deduplication or model fitting.',
          '- CSV parser/schema failures stop execution rather than silently skipping malformed records.',
          '', 'Detailed CSV tables and runtime/snapshot manifest: `data-profiling/`. '
          'See `../docs/data-source.md` for unresolved source provenance.']
(REPORT_DIR / CONFIG['output']['quality_report_filename']).write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'Exported {len(tables)} tables, manifest and quality report to {REPORT_DIR}')